This is a short script that may be used to find the strong symmetries in an open equantum system, subject to jump operators $L$ and Hamiltonian evolution $H$. 

We give its implementation for our systems of interest below

In [ ]:
import sympy as sp
import numpy as np

Below we define the spin-1/2 system 

In [ ]:

Delta, Delta1, Delta2, Omega, phi = sp.symbols('Delta Delta_1 Delta_2 Omega phi', real=True)

# Pauli matrices
sx = sp.Matrix([[0, 1], [1, 0]])
sy = sp.Matrix([[0, -sp.I], [sp.I, 0]])
sz = sp.Matrix([[1, 0], [0, -1]])
I2 = sp.eye(2)

# Spin operators
Sp = (sx + sp.I*sy) / 2
Sm = (sx - sp.I*sy) / 2

In [ ]:
Cm = sp.symbols('C')

# Two-spin operators via Kronecker product
S1z = sp.kronecker_product(sz, I2)
S1x = sp.kronecker_product(sx, I2)
S1y = sp.kronecker_product(sy, I2)
S2z = sp.kronecker_product(I2, sz)
S2x = sp.kronecker_product(I2, sx)
S2y = sp.kronecker_product(I2, sy)
S1p = sp.kronecker_product(Sp, I2)
S2p = sp.kronecker_product(I2, Sp)
S1m = sp.kronecker_product(Sm, I2)
S2m = sp.kronecker_product(I2, Sm)
I4  = sp.eye(4)

H = Delta/2 * S1z + Delta/2*S2z + Omega * (S1p + sp.exp(sp.I*phi)*S2p + S1m + sp.exp(-sp.I*phi)*S2m) + Cm/2 * (S1p*S1m + S2p*S2m + S1p*S2m*sp.exp(-sp.I*phi) + S2p*S1m*sp.exp(sp.I*phi))

L = S1m + sp.exp(-sp.I*phi)*S2m

In [ ]:
x = sp.symbols('x00:16')  # x00..x15
X = sp.Matrix([[x[0], x[1], x[2], x[3]],
               [x[4], x[5],x[6], x[7]],
               [x[8], x[9], x[10], x[11]], 
               [x[12], x[13], x[14], x[15]]])

# Constraints: [X,H] = 0 and [X,L] = 0
comm_H = sp.simplify(X*H - H*X)
comm_L = sp.simplify(X*L - L*X)

# Flatten constraints to a list of scalar equations
eqs = list(comm_H)
eqs.extend(list(comm_L))
A, b = sp.linear_eq_to_matrix(eqs, list(x))

# Compute nullspace of A (solutions for vec(x))
ns = A.nullspace()


# Turn each nullspace vector into a 3x3 matrix and simplify
basis_matrices = []
for idx, v in enumerate(ns):
    M = sp.Matrix(4,4, list(v))   # shape into 3x3
    M = sp.simplify(M)
    basis_matrices.append(M)



I = sp.eye(4)
B = None
for M in basis_matrices:
    if sp.simplify(M - I) == sp.zeros(3):
        # this is the identity basis
        continue
    else:
        # choose the first non-identity as B (if there is one)
        if B is None:
            B = M
B

The output `B` is the strong symmetry operator, defined as $\mathbf{S}^2_\varphi$ in the main text. 

We can write out its eigendecomposition to make it into a more understandable form

In [ ]:
# Strong symmetry generator eigenvectors (akin to Ang. Momentum eigenstates)
B.eigenvects()

For the three level system we instead define the following Lindbladian

In [ ]:
# Kets (column vectors) in the basis |0>,|1>,|2>
ket0 = sp.Matrix([1, 0, 0])
ket1 = sp.Matrix([0, 1, 0])
ket2 = sp.Matrix([0, 0, 1])

# Bras (conjugate transpose of the kets)
bra0 = ket0.conjugate().T
bra1 = ket1.conjugate().T
bra2 = ket2.conjugate().T

# Projectors and transitions
P1 = ket1 * bra1   # |1><1|
P2 = ket2 * bra2   # |2><2|
T01 = ket0 * bra1  # |0><1|
T02 = ket0 * bra2  # |0><2|

# Hamiltonian 
H = sp.simplify(Delta * P1 + Delta * P2 +  Omega * T01 + Omega * sp.exp(-sp.I*phi) * T02 + ( Omega * T01 + Omega * sp.exp(-sp.I*phi) * T02).conjugate().T + Cm/2 * (T01.conjugate()*T01 + T01.conjugate()*T02*sp.exp(-sp.I*phi)+ T02.conjugate()*T01*sp.exp(sp.I*phi) + T02.conjugate()*T02)) 

L = sp.simplify(T01 + sp.exp(-sp.I*phi) * T02)

In [ ]:
ket0 = sp.Matrix([1,0,0]); ket1 = sp.Matrix([0,1,0]); ket2 = sp.Matrix([0,0,1])
bra0 = ket0.conjugate().T; bra1 = ket1.conjugate().T; bra2 = ket2.conjugate().T


P1 = ket1*bra1    # |1><1|
P2 = ket2*bra2    # |2><2|
T01 = ket0*bra1   # |0><1|
T02 = ket0*bra2   # |0><2|
T10 = ket1*bra0   # |1><0|
T20 = ket2*bra0   # |2><0|

# # Hamiltonian 
H = sp.simplify(Delta * P1 + Delta * P2
             + Omega * T01 + Omega * sp.exp(-sp.I*phi) * T02
             + Omega * T10 + Omega * sp.exp(sp.I*phi) * T20)

L = sp.simplify(T01 + sp.exp(-sp.I*phi) * T02)

# Unknown operator X with 9 symbolic entries
x = sp.symbols('x00:09')  # x00..x08
X = sp.Matrix([[x[0], x[1], x[2]],
               [x[3], x[4], x[5]],
               [x[6], x[7], x[8]]])

# Constraints: [X,H] = 0 and [X,L] = 0
comm_H = sp.simplify(X*H - H*X)
comm_L = sp.simplify(X*L - L*X)

# Flatten constraints to a list of scalar equations
eqs = list(comm_H)
eqs.extend(list(comm_L))
A, b = sp.linear_eq_to_matrix(eqs, list(x))

# Compute nullspace of A (solutions for vec(x))
ns = A.nullspace()

# print("A matrix rank:", A.rank())
# print("Dimension of nullspace:", len(ns))
# print()

# Turn each nullspace vector into a 3x3 matrix and simplify
basis_matrices = []
for idx, v in enumerate(ns):
    M = sp.Matrix(3,3, list(v))   # shape into 3x3
    M = sp.simplify(M)
    basis_matrices.append(M)
    # print(f"Basis matrix #{idx+1}:")
    # sp.pprint(M)
    # print()

# Identify common-case ordering (one basis vector is identity)
# Find identity-like vector if present
I = sp.eye(3)
B = None
for M in basis_matrices:
    if sp.simplify(M - I) == sp.zeros(3):
        # this is the identity basis
        continue
    else:
        # choose the first non-identity as B (if there is one)
        if B is None:
            B = M
B

Where again `B` gives us the strong symmetry $\mathcal{T}_\varphi$ defined in the main text

The eigenvectors defined in the main text can be extracted, note the introduction of the global unitary here is to simplify the eigenmode notation.

In [ ]:
Bn = B * sp.exp(sp.I*phi) 
Bn.eigenvects()

### Additional Remarks


Once we find a strong symmetry operator, we may use it to block diagonalize the full Liouvillian. 

We consider below performing this for the three level system.

In [ ]:

# bare kets (column vectors)
ket0 = sp.Matrix([1,0,0])
ket1 = sp.Matrix([0,1,0])
ket2 = sp.Matrix([0,0,1])

# bras
bra0 = ket0.conjugate().T
bra1 = ket1.conjugate().T
bra2 = ket2.conjugate().T

# projectors and transitions (bare basis)
P1 = ket1 * bra1    # |1><1|
P2 = ket2 * bra2    # |2><2|
T01 = ket0 * bra1   # |0><1|
T02 = ket0 * bra2   # |0><2|
T10 = ket1 * bra0   # |1><0|
T20 = ket2 * bra0   # |2><0|

# phi = 0 case: exp(i phi) = 1
H = sp.simplify(Delta * P1 + Delta * P2
                + Omega * T01 + Omega * T02
                + Omega * T10 + Omega * T20)

L = sp.simplify(T01 + T02)   # since phi = 0

# New basis vectors (columns of U)
e0p   = ket0
eplus = (ket1 + ket2) / sp.sqrt(2)
eminus= (ket1 - ket2) / sp.sqrt(2)

# Build unitary U whose columns are the new basis (in original ordering)
U = sp.Matrix.hstack(e0p, eplus, eminus)

# Transform operators to new basis: O' = U^\dagger O U
Udag = U.conjugate().T
Hp = sp.simplify(Udag * H * U)
Lp = sp.simplify(Udag * L * U)


In this rotated basis we can block-diagonalize the Liouvillian, exploiting its vectorized form $||\mathcal{L}\rangle\rangle$

In [ ]:
gamma = sp.symbols('Gamma', real=True, positive=True)   # decay rate

# --- Lindblad map on a symbolic rho (3x3) ---
r11,r12,r13,r21,r22,r23,r31,r32,r33 = sp.symbols(
    'r11 r12 r13 r21 r22 r23 r31 r32 r33'
)
rho = sp.Matrix([[r11,r12,r13],
                 [r21,r22,r23],
                 [r31,r32,r33]])

comm = -1j * (Hp * rho - rho * Hp)
diss = gamma * (Lp * rho * Lp.H - sp.Rational(1,2) * (Lp.H * Lp * rho + rho * Lp.H * Lp))
Lindblad_map = sp.simplify(comm + diss)   # 3x3 matrix expression 

# --- Vectorized Liouvillian (column-major vec) ---
Id = sp.eye(n)
# -i[H, .] -> -i*(I ⊗ H - H.T ⊗ I)
Ham_super = -1j * (sp.kronecker_product(Id, Hp) - sp.kronecker_product(Hp.T, Id))

# vec(L rho L^†) = ( (L^†).T ⊗ L ) vec(rho) = (L.conjugate() ⊗ L) vec(rho)
L_term = sp.kronecker_product(Lp.conjugate(), Lp)
LdL = (Lp.H * Lp)
Diss_super = gamma * ( L_term
                      - sp.Rational(1,2) * ( sp.kronecker_product(Id, LdL)
                                            + sp.kronecker_product(LdL.T, Id) ) )

Liouv = sp.simplify(Ham_super + Diss_super)   # 9x9

# --- Sanity check: Liouv * vec(rho) == vec(Lindblad_map) ---
vec_rho = sp.Matrix([r11, r21, r31, r12, r22, r32, r13, r23, r33])  # column-major vec
vec_map_colmaj = sp.Matrix([
    Lindblad_map[0,0], Lindblad_map[1,0], Lindblad_map[2,0],
    Lindblad_map[0,1], Lindblad_map[1,1], Lindblad_map[2,1],
    Lindblad_map[0,2], Lindblad_map[1,2], Lindblad_map[2,2]
])

check = sp.simplify(Liouv * vec_rho - vec_map_colmaj)

The Liouvillian `Liouv` is now block-diagonalized and its entries can be analyzed individually.

With this we may address features like initial state overlaps and decoherence-free subspaces of the individual eigenmodes of $\mathcal{L}$ without needing to run the dynamics

For example to obtain the overlap of some `rho` with a particular eigenmode of `Liouv`, we can find

In [ ]:
# Replace Delta with 0 (simplicity)
Liouv0 = Liouv.subs(Delta, 0)
# Stack into column-major vec
m_0 = sp.Matrix([1,0,0,0,0,0,0,0,0])
m_bright= sp.Matrix([0,0,0,0,1,0,0,0,0])
m_dark = sp.Matrix([0,0,0,0,0,0,0,0,1])

R_eig = Liouv0.eigenvects()
L_eig = Liouv0.H.eigenvects()

In [ ]:
# Example density matrix
a = 0.8
b = 0.5
c = sp.sqrt(1-a**2-b**2)
psi = sp.Matrix([a,b,c])
rho0 = psi * psi.H

L_eig_k = sp.Matrix(3,3, list(L_eig[3][2][0]))   
# Notation: k,index,vecs = L_eig where k is the k-th eigenvalue (k=0,1,...,dim(Liov))
# , index=(eigenvalue,m,eigenvect_list) 
# and v the corresponding eigenvector in eigvlist

# sub in arbitrary decay and drive rates to obtain a numerical value
sp.trace(L_eig_k.H * rho0).subs({gamma:1, Omega:1})